# 🐝 Lab 1.3 — Explain & Dethrone: SHAP vs Tabular Foundation Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab03_shap_tabpfn.ipynb)

**Gradient Hive: Advanced ML Week 2026 · Day 1 — Classical, Sharpened**

⏱ **Time:** ~1.5 hours · runs on CPU (Colab free tier is plenty)

### 🎯 Learning objectives
- Derive Shapley values from cooperative game theory and **implement exact Shapley values by enumerating coalitions**.
- Verify your implementation against `shap.TreeExplainer`, and estimate Shapley values by **permutation sampling**.
- Read **beeswarm, waterfall, dependence and interaction** plots, and recognise the **pitfalls** (correlated features, off-manifold evaluation, 'SHAP ≠ causality').
- Explain how **TabPFN** (a prior-fitted transformer) does tabular prediction by *in-context learning*.
- Benchmark TabPFN against **tuned gradient-boosted trees** on small datasets, and decide when to use which.

### 🧰 Prerequisites
- Lab 1.1 (gradient boosting) or comfort with scikit-learn tree ensembles.
- Basic combinatorics (subsets, factorials) and a rough idea of what a transformer is.

### 🌍 Why this matters in 2026
Two forces are reshaping tabular ML. **Regulation and trust:** credit, insurance and hiring models must be explained, and SHAP has become the de facto explanation format, so you need to know exactly what those numbers do and don't mean. **Foundation models arrive for spreadsheets:** TabPFN v2 (Hollmann et al., *Nature* 2025) showed that a transformer pre-trained only on *synthetic* datasets can match or beat tuned GBDTs on small tables in a single forward pass, with no hyperparameter search. Knowing when the new kid wins, and when your trusty booster still rules, is a 2026 core skill.

> 🐝 *Buzz the Backprop Bee says:* run every cell top to bottom. Each **🧪 Your Turn** cell is followed by a **✅ Solution** cell, so the notebook always runs end to end. Try the exercise first, then peek.

## 🗺️ Roadmap

| Part | What you'll do | ⏱ min |
|---|---|---|
| ⚙️ Setup | imports, seeds, device | 3 |
| 1 · Shapley from scratch | value function, exact enumeration, permutation sampling | 20 |
| 2 · SHAP in practice | waterfall, beeswarm, dependence on California housing | 10 |
| 3 · Interactions | SHAP interaction values; recover a planted x0·x1 | 8 |
| 4 · Pitfalls | off-manifold hybrids, twin features, causality | 12 |
| 5 · TabPFN | in-context learning; decision-surface grid; benchmark vs tuned GBDT | 20 |
| 6 · Leaderboard | 🍯 Honey Yield challenge: beat the baselines | 12 |
| 7 · Field guide | when to use which | 5 |

## ⚙️ Setup

In [ ]:
# ⚙️ Setup: run me first!
import os, sys, time, math, random, itertools, warnings, importlib.util
warnings.filterwarnings("ignore")   # silence tqdm/IProgress + CPU-size warnings from shap/tabpfn
FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode: shrinks everything

def ensure(pkgs):
    """pip-install packages that are missing (shap is preinstalled on Colab; tabpfn is not)."""
    missing = [pip for pip, mod in pkgs if importlib.util.find_spec(mod) is None]
    if missing:
        print("📦 Installing:", missing)
        get_ipython().run_line_magic("pip", "install -q " + " ".join(missing))

ensure([("shap", "shap")])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import shap
from scipy.stats import loguniform, randint
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.datasets import load_diabetes, load_breast_cancer, load_wine, load_digits, make_classification
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold, RandomizedSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, SplineTransformer
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss, r2_score
warnings.filterwarnings("ignore")

SEED = 42
N_JOBS = 2          # parallel workers for sklearn searches (Colab free tier has 2 vCPUs)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA", "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
print(f"FAST={FAST} | device={DEVICE} | shap {shap.__version__}")

---
## Part 1 — Shapley values from first principles

**The game.** $d$ players (features) cooperate to produce a payout (the model's prediction for one input $x$). How should we split the payout *fairly*? Lloyd Shapley (1953) proved there's exactly **one** way to do it that satisfies four axioms:

| Axiom | Meaning for features |
|---|---|
| **Efficiency** | attributions add up: $\sum_i \phi_i = f(x) - \mathbb{E}[f(X)]$ |
| **Symmetry** | two features that contribute identically get equal credit |
| **Dummy** | a feature that never changes the output gets 0 |
| **Additivity** | for a model $f+g$, attributions add up |

The unique solution averages feature $i$'s **marginal contribution** over all coalitions $S$ of the other features:

$$\phi_i = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!\,(d-|S|-1)!}{d!}\Big[v(S \cup \{i\}) - v(S)\Big].$$

The weight is the probability that, in a uniformly random ordering of the $d$ features, exactly the features in $S$ come before $i$.

**The value function** $v(S)$ is "the prediction when only the features in $S$ are known". SHAP's *interventional* (a.k.a. marginal) choice fills the unknown features from a **background dataset**:
$$v(S) = \mathbb{E}_{\tilde X \sim \text{background}}\big[f(x_S, \tilde X_{\bar S})\big].$$

That's $2^d$ coalitions. Fine for $d = 6$ (64), hopeless for $d = 50$ ($10^{15}$). Keep that in mind; it's why TreeSHAP exists.

In [ ]:
FEATS = ["age", "bmi", "bp", "s1", "s5", "s6"]
diab = load_diabetes(as_frame=True)
Xd = diab.data[FEATS].astype(np.float32)          # float32: shap evaluates trees in float32 (see tip below)
yd = diab.target.values
tiny = HistGradientBoostingRegressor(max_iter=100, max_depth=3, random_state=SEED).fit(Xd, yd)
background = Xd.iloc[:50].values
x_explain = Xd.iloc[100:110].values
f = tiny.predict
print(f"tiny model R² (train) = {r2_score(yd, f(Xd)):.3f}; d = {len(FEATS)} features -> {2 ** len(FEATS)} coalitions")

> 💡 **Tip (a real-world gotcha):** `shap` evaluates trees in **float32**. If you train on float64 features, a value sitting exactly next to a split threshold can fall on different sides in sklearn vs shap, and the "additivity check" fails. Casting features to `float32` before fitting avoids this.

### 🧪 Your Turn 1 — The value function
Implement `value(f, x, S, background)`. Copy the background, overwrite the columns in `S` with `x`'s values, predict, and average.

In [ ]:
# 🧪 Your Turn 1
def value(f, x, S, background):
    Z = background.copy()
    # TODO: set the columns listed in S to x's values (hint: Z[:, list(S)] = ...)
    return None   # TODO: mean prediction over the hybrid rows

In [ ]:
# ✅ Solution (peek only after trying!)
def value(f, x, S, background):
    Z = background.copy()
    S = list(S)
    if S:
        Z[:, S] = x[S]
    return float(f(Z).mean())

### 🧪 Your Turn 2 — Exact Shapley by enumeration
Loop over every feature $i$ and every subset $S$ of the *other* features (`itertools.combinations`), weight the marginal contribution, and add it up.

*Hint:* `math.factorial(len(S)) * math.factorial(d - len(S) - 1) / math.factorial(d)`. Cache `value` results in a dict keyed by `frozenset(S)`, because each coalition is reused many times.

In [ ]:
# 🧪 Your Turn 2
def exact_shapley(f, x, background):
    d = len(x)
    phi = np.zeros(d)
    for i in range(d):
        others = [j for j in range(d) if j != i]
        for r in range(d):
            for S in itertools.combinations(others, r):
                w = None      # TODO: Shapley weight
                gain = None   # TODO: v(S ∪ {i}) - v(S)
                # TODO: phi[i] += w * gain
    return phi

In [ ]:
# ✅ Solution (peek only after trying!)
def exact_shapley(f, x, background):
    d = len(x)
    cache = {}
    def v(S):
        key = frozenset(S)
        if key not in cache:
            cache[key] = value(f, x, S, background)
        return cache[key]
    phi = np.zeros(d)
    for i in range(d):
        others = [j for j in range(d) if j != i]
        for r in range(d):
            for S in itertools.combinations(others, r):
                w = math.factorial(r) * math.factorial(d - r - 1) / math.factorial(d)
                phi[i] += w * (v(S + (i,)) - v(S))
    return phi

In [ ]:
t0 = time.time()
phi_ours = np.array([exact_shapley(f, x, background) for x in x_explain])
t_ours = time.time() - t0
t0 = time.time()
tree_expl = shap.TreeExplainer(tiny, data=background, feature_perturbation="interventional")
phi_shap = tree_expl.shap_values(x_explain)
t_shap = time.time() - t0
base = f(background).mean()

# ✔️ Checkpoint
eff_err = np.abs(phi_ours.sum(1) - (f(x_explain) - base)).max()
match_err = np.abs(phi_ours - phi_shap).max()
print(f"efficiency error {eff_err:.2e} | max |ours - TreeExplainer| = {match_err:.2e}")
print(f"time: enumeration {t_ours:.2f}s vs TreeSHAP {t_shap:.3f}s for {len(x_explain)} rows")
assert eff_err < 1e-6, "Shapley values must sum to f(x) - E[f]"
assert match_err < 1e-3, "should match shap's interventional TreeExplainer"
print("✔️ Your brute-force Shapley values match shap.TreeExplainer exactly! 🐝")
pd.DataFrame(phi_ours[:4], columns=FEATS, index=[f"row {i}" for i in range(4)]).round(2)

**Why is TreeSHAP so much faster?** For tree ensembles, Lundberg et al. (2020) showed that the sum over $2^d$ coalitions can be computed in polynomial time by pushing *all subsets at once* down each tree. That's $O(T L D^2)$ for path-dependent TreeSHAP ($T$ trees, $L$ leaves, depth $D$), and linear in the background size for the interventional variant. For arbitrary black boxes there's no such trick, so we **sample**.

### 🧪 Your Turn 3 — Permutation-sampling Shapley (Štrumbelj & Kononenko, 2014)
Shapley values are the average marginal contribution over random **orderings**. Sample `n_perm` permutations. For each one, add features one at a time and credit each feature with the change in $v$. Efficiency holds exactly for *every* permutation. Can you see why?

In [ ]:
# 🧪 Your Turn 3
def sampling_shapley(f, x, background, n_perm, rng):
    d = len(x); phi = np.zeros(d)
    for _ in range(n_perm):
        perm = rng.permutation(d)
        S, prev = [], value(f, x, [], background)
        for i in perm:
            pass  # TODO: add i to S, compute the new value, credit phi[i] with the change, update prev
    return phi / n_perm

In [ ]:
# ✅ Solution (peek only after trying!)
def sampling_shapley(f, x, background, n_perm, rng):
    d = len(x); phi = np.zeros(d)
    for _ in range(n_perm):
        perm = rng.permutation(d)
        S, prev = [], value(f, x, [], background)
        for i in perm:
            S.append(i)
            cur = value(f, x, S, background)
            phi[i] += cur - prev
            prev = cur
    return phi / n_perm

In [ ]:
rng = np.random.default_rng(SEED)
x0 = x_explain[0]
budgets = [1, 3, 10, 30, 100] if FAST else [1, 3, 10, 30, 100, 300]
errs = []
for m in budgets:
    trials = [np.abs(sampling_shapley(f, x0, background, m, rng) - phi_ours[0]).mean() for _ in range(3 if FAST else 8)]
    errs.append(np.mean(trials))
est = sampling_shapley(f, x0, background, 5, rng)
# ✔️ Checkpoint: efficiency holds for any number of permutations; error shrinks with budget
assert abs(est.sum() - (f(x0[None])[0] - base)) < 1e-6
assert errs[-1] < errs[0]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.loglog(budgets, errs, "o-", color=HIVE[2], label="sampling error")
ax.loglog(budgets, errs[0] / np.sqrt(np.array(budgets)), "--", color="gray", label=r"$\propto 1/\sqrt{m}$")
ax.set_xlabel("# permutations m"); ax.set_ylabel("mean |φ̂ - φ|"); ax.set_title("Monte Carlo Shapley converges like 1/√m"); ax.legend()
plt.show()
print("✔️ Sampling Shapley: unbiased, exactly efficient, and converging.")

> **🍯 Key takeaway:** Shapley values are the *unique* fair split of $f(x) - \mathbb{E}[f]$ satisfying efficiency, symmetry, dummy and additivity. Exact computation costs $2^d$ evaluations, sampling converges like $1/\sqrt{m}$, and TreeSHAP is exact and fast for trees.

---
## Part 2 — SHAP in practice: beeswarm, waterfall, dependence

Now a real model on California housing: a `HistGradientBoostingRegressor` predicting median house value (in \$100k). We'll explain 1,000 test districts with an interventional TreeExplainer and a 100-row background.

How to read the plots:
* **Waterfall** (one prediction): start at $\mathbb{E}[f(X)]$, and each feature pushes the prediction up (red) or down (blue) until you reach $f(x)$. That's the efficiency axiom, drawn.
* **Beeswarm** (global): one dot per row per feature. The x-position is the SHAP value and the colour is the feature value. Features are sorted by mean $|\phi|$.
* **Dependence / scatter**: SHAP value of one feature against its value. Vertical spread at a fixed $x$ hints at **interactions**.

In [ ]:
def load_housing():
    try:
        from sklearn.datasets import fetch_california_housing
        d = fetch_california_housing(as_frame=True)
        return d.data.astype(np.float32), d.target.values, "California housing"
    except Exception as e:
        print("⚠️ download failed:", e, "-> using diabetes data instead")
        d = load_diabetes(as_frame=True)
        return d.data.astype(np.float32), d.target.values, "diabetes"

Xh, yh, HNAME = load_housing()
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.2, random_state=SEED)
if FAST:
    Xh_tr, yh_tr = Xh_tr.iloc[:4000], yh_tr[:4000]
house = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, random_state=SEED).fit(Xh_tr, yh_tr)
print(f"{HNAME}: test R² = {r2_score(yh_te, house.predict(Xh_te)):.3f}")

bg = shap.sample(Xh_tr, 100, random_state=SEED)
explainer = shap.TreeExplainer(house, data=bg, feature_perturbation="interventional")
X_ex = Xh_te.iloc[: 200 if FAST else 1000]
sv = explainer(X_ex)           # a shap.Explanation: .values, .base_values, .data, .feature_names
print("SHAP values:", sv.values.shape)

In [ ]:
shap.plots.waterfall(sv[0], max_display=9, show=False)
plt.title("Why did the model predict this value for district #0?"); plt.tight_layout(); plt.show()

In [ ]:
shap.plots.beeswarm(sv, max_display=10, show=False)
plt.title("Global view: every dot is one district"); plt.tight_layout(); plt.show()

In [ ]:
top_feat = X_ex.columns[np.argmax(np.abs(sv.values).mean(0))]
shap.plots.scatter(sv[:, top_feat], color=sv, show=False)   # colour = feature shap picks as strongest interaction
plt.title(f"Dependence plot for {top_feat}"); plt.tight_layout(); plt.show()

> **🍯 Key takeaway:** Waterfall = one prediction's story; beeswarm = global importance *and* direction; dependence plots reveal shapes and hint at interactions.

---
## Part 3 — Interaction effects

> 🔮 **Predict-then-run:** for California housing, which *pair* of features do you expect to interact most strongly? (Think about what a single number like "latitude" can and can't tell you about location.)

Shapley values split $f(x) - \mathbb{E}f$ among single features. **SHAP interaction values** (Lundberg et al., 2020) go one level deeper. They split it into a matrix $\Phi_{ij}$, with main effects on the diagonal and pairwise interactions off it (each interaction split half-and-half between $\Phi_{ij}$ and $\Phi_{ji}$), so $\sum_{j} \Phi_{ij} = \phi_i$. They're based on the Shapley interaction index:

$$\Phi_{ij} = \sum_{S \subseteq N\setminus\{i,j\}} \frac{|S|!\,(d-|S|-2)!}{2\,(d-1)!}\Big[v(S\cup\{i,j\}) - v(S\cup\{i\}) - v(S\cup\{j\}) + v(S)\Big], \quad i \neq j.$$

Sanity check first on a synthetic function where we *know* the answer: $y = x_0 x_1 + x_2 + \varepsilon$, with $x_3$ irrelevant.

In [ ]:
r = np.random.default_rng(0)
Xs = r.normal(size=(3000, 4)).astype(np.float32)
ys = Xs[:, 0] * Xs[:, 1] + Xs[:, 2] + 0.1 * r.normal(size=3000)
syn = HistGradientBoostingRegressor(max_iter=200, random_state=SEED).fit(Xs, ys)
inter = shap.TreeExplainer(syn).shap_interaction_values(Xs[:300])     # path-dependent TreeSHAP
M = np.abs(inter).mean(0)
off = M.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
# ✔️ Checkpoint: the strongest interaction is (x0, x1); rows of the matrix sum to the SHAP values
assert {i, j} == {0, 1}
phi_path = shap.TreeExplainer(syn).shap_values(Xs[:300])
assert np.allclose(inter.sum(2), phi_path, atol=1e-4)
print(f"✔️ Strongest interaction found: x{i} × x{j} (true model: x0·x1)")

Xi = Xh_te.iloc[: 150 if FAST else 400]
inter_h = shap.TreeExplainer(house).shap_interaction_values(Xi)
Mh = np.abs(inter_h).mean(0)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.5]})
for ax, mat, names, title in [(axes[0], M, [f"x{k}" for k in range(4)], "synthetic: y = x0·x1 + x2"),
                              (axes[1], Mh, list(Xh.columns), HNAME)]:
    im = ax.imshow(mat, cmap="YlOrBr"); ax.set_xticks(range(len(names)), names, rotation=45, ha="right"); ax.set_yticks(range(len(names)), names)
    ax.set_title("mean |SHAP interaction|: " + title, fontsize=10); plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()
offh = Mh.copy(); np.fill_diagonal(offh, 0); a, b = np.unravel_index(offh.argmax(), offh.shape)
print(f"Strongest interaction in {HNAME}: {Xh.columns[a]} × {Xh.columns[b]}")

For California housing the top interaction is usually **Latitude × Longitude**. Location is a 2-D concept, and a tree can only describe it by combining the two coordinates. A model that looked "additive" in the beeswarm is anything but.

> **🍯 Key takeaway:** Interaction values decompose each SHAP value into a main effect plus pairwise interactions. On California housing the top pair is typically the location pair Latitude × Longitude.

---
## Part 4 — Pitfalls: correlated features, impossible inputs, and causality 🕳️

**Pitfall 1: correlated features split the credit.** If $x_0$ and a near-copy $x_0'$ both exist, the model can use either. Their SHAP values divide the credit somewhat arbitrarily, depending on which one the trees happened to split on. Each feature then looks *less* important than the underlying signal is.

**Pitfall 2: off-manifold evaluation.** The interventional value function glues $x_S$ onto *random* background rows. With correlated features this creates inputs that never occur in reality, and the model's behaviour there is untested extrapolation. Below we take a coastal district's **latitude** and pair it with random background **longitudes**.

**Pitfall 3: SHAP explains the *model*, not the *world*.** A big SHAP value means the model *uses* the feature, not that changing it *causes* a change in $y$ (Janzing et al., 2020; Kumar et al., 2020). Whether to be "true to the model" (interventional) or "true to the data" (conditional/observational) is a modelling choice (Chen et al., 2020).

In [ ]:
# Pitfall 2, visualised: hybrid rows used by the interventional value function for S = {Latitude}
if "Latitude" in Xh.columns:
    x_star = Xh_te.iloc[np.argmin(np.abs(Xh_te["Longitude"] + 122.4) + np.abs(Xh_te["Latitude"] - 37.8))]  # near San Francisco
    hybrid_lat = np.full(len(bg), x_star["Latitude"]); hybrid_lon = bg["Longitude"].values
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.scatter(Xh_tr["Longitude"][:5000], Xh_tr["Latitude"][:5000], s=2, color="gray", alpha=0.3, label="real districts")
    ax.scatter(hybrid_lon, hybrid_lat, s=18, color=HIVE[2], label="hybrid rows v({Latitude}) evaluates")
    ax.scatter([x_star["Longitude"]], [x_star["Latitude"]], s=150, marker="*", color=HIVE[0], edgecolor="k", label="explained district")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude"); ax.legend(loc="lower left", fontsize=9)
    ax.set_title("Off-manifold: hybrid rows land in the ocean and in empty land")
    plt.tight_layout(); plt.show()

In [ ]:
# Pitfall 1: the twin experiment
r = np.random.default_rng(1)
n = 3000
x_sig = r.normal(size=n)
base_X = np.c_[x_sig, r.normal(size=(n, 2))]
y_tw = 2 * x_sig + base_X[:, 1] + 0.3 * r.normal(size=n)
X_single = pd.DataFrame(base_X, columns=["signal", "other", "noise"]).astype(np.float32)
X_twin = X_single.assign(signal_twin=(x_sig + 0.02 * r.normal(size=n)).astype(np.float32))

def mean_abs_shap(X):
    mdl = HistGradientBoostingRegressor(max_iter=200, random_state=SEED).fit(X, y_tw)
    ex = shap.TreeExplainer(mdl, data=X.iloc[:100], feature_perturbation="interventional")
    return ex.shap_values(X.iloc[:800]), mdl

sv_single, _ = mean_abs_shap(X_single)
sv_twin, _ = mean_abs_shap(X_twin)
imp = pd.DataFrame({"without twin": pd.Series(np.abs(sv_single).mean(0), X_single.columns),
                    "with twin": pd.Series(np.abs(sv_twin).mean(0), X_twin.columns)}).fillna(0)
imp.plot.barh(color=[HIVE[3], HIVE[2]], figsize=(7, 3.5), title="mean |SHAP|: adding a near-duplicate splits the credit")
plt.xlabel("mean |SHAP|"); plt.tight_layout(); plt.show()
display(imp.round(3))

### 🧪 Your Turn 4 — Grouped attributions
Because SHAP values are additive, you can **sum** the SHAP values of a group of correlated features to get one group attribution per row. (This is the efficiency axiom applied to a "team"; it is a practical approximation, not the Shapley value of the group as a single player.) Compute the mean |group SHAP| for `{signal, signal_twin}` and compare it with `signal` in the model without the twin.

In [ ]:
# 🧪 Your Turn 4
cols = list(X_twin.columns)
group_attr = None   # TODO: per-row sum of SHAP values of "signal" and "signal_twin" (use sv_twin and cols.index)
group_importance = None   # TODO: mean absolute value of group_attr

In [ ]:
# ✅ Solution (peek only after trying!)
cols = list(X_twin.columns)
group_attr = sv_twin[:, cols.index("signal")] + sv_twin[:, cols.index("signal_twin")]
group_importance = np.abs(group_attr).mean()

In [ ]:
# ✔️ Checkpoint: group attribution vs the single-feature model
single_importance = np.abs(sv_single[:, 0]).mean()
print(f"group (signal + twin) = {group_importance:.3f} vs signal alone (no twin) = {single_importance:.3f}")
# ✔️ Checkpoint
assert abs(group_importance - single_importance) / single_importance < 0.05
print("✔️ Grouping recovers the full importance of the underlying signal.")

> **🍯 Key takeaway:** SHAP explains the *model*, not the world. Correlated features split credit (sum them into groups), interventional SHAP queries impossible inputs, and none of it is causal.

---
## Part 5 — The 2026 twist: TabPFN, a foundation model for tables

**Idea: learn the learning algorithm.** A *Prior-data Fitted Network* (Müller et al., 2022) is a transformer trained once, offline, on **millions of synthetic datasets**. Each one is drawn from a prior over data-generating processes, such as random structural causal models with noise, nonlinearities and missingness. Each training example is a whole dataset. The network sees $(X_{\text{train}}, y_{\text{train}}, X_{\text{test}})$ and must predict $y_{\text{test}}$.

After pre-training, **"fitting" on your data is just a forward pass**. Your training table becomes the *context* (in-context learning, like few-shot prompting an LLM), and predictions approximate the **Bayesian posterior predictive** under that synthetic prior:

$$q_\theta(y_{\text{test}} \mid x_{\text{test}}, D_{\text{train}}) \;\approx\; p(y_{\text{test}} \mid x_{\text{test}}, D_{\text{train}}) = \int p(y \mid x, \varphi)\, p(\varphi \mid D_{\text{train}})\, d\varphi.$$

**TabPFN v2** (Hollmann et al., *Nature* 2025) uses a two-way attention architecture (across features *and* across samples). It handles classification and regression, missing values and categorical features, and it was designed for datasets up to about 10k rows and 500 features. In the paper it beat heavily tuned baselines on small-data benchmarks in seconds.

**Practicalities on CPU** ⚠️: the cost of the forward pass grows with (train rows × test rows × features), so on CPU we stick to **≤ 1,000 rows** and a few ensemble members. Newer checkpoints (TabPFN-2.5 and later) need a one-time licence acceptance and a `TABPFN_TOKEN` environment variable. By default we use the openly downloadable **v2** weights. Everything below is wrapped so the lab still runs if TabPFN can't be installed, downloaded, or is too slow.

> ⚡ **GPU recommended** for this part. On a Colab T4 TabPFN is roughly 10–50× faster; on CPU expect a few minutes in total.

In [ ]:
TABPFN_OK, TABPFN_MSG = False, ""
N_ENS = 8 if DEVICE == "cuda" else (2 if FAST else 4)     # TabPFN internal ensemble size

def make_tabpfn(kind="clf"):
    from tabpfn import TabPFNClassifier, TabPFNRegressor
    cls = TabPFNClassifier if kind == "clf" else TabPFNRegressor
    if os.environ.get("TABPFN_TOKEN"):                   # licensed latest checkpoint
        return cls(device=DEVICE, n_estimators=N_ENS)
    try:                                                 # open TabPFN v2 weights (Nature 2025 model)
        from tabpfn.constants import ModelVersion
        return cls.create_default_for_version(ModelVersion.V2, device=DEVICE, n_estimators=N_ENS)
    except (ImportError, AttributeError):                # older tabpfn 2.x releases default to v2
        return cls(device=DEVICE, n_estimators=N_ENS)

if os.environ.get("SKIP_TABPFN") == "1":
    TABPFN_MSG = "SKIP_TABPFN=1 set"
else:
    try:
        ensure([("tabpfn", "tabpfn")])
        Xb_, yb_ = load_breast_cancer(return_X_y=True)
        t0 = time.time()
        _m = make_tabpfn("clf").fit(Xb_[:200], yb_[:200])
        _acc = (_m.predict(Xb_[200:300]) == yb_[200:300]).mean()
        dt = time.time() - t0
        TABPFN_OK = dt < 240                             # includes the one-off weight download
        TABPFN_MSG = f"smoke test ok: acc={_acc:.2f} in {dt:.1f}s" if TABPFN_OK else f"too slow ({dt:.0f}s) on this machine"
    except Exception as e:
        TABPFN_MSG = f"{type(e).__name__}: {str(e)[:300]}"

print(("✅ TabPFN ready: " if TABPFN_OK else "⚠️ TabPFN unavailable, continuing with GBDT/linear baselines only. Reason: ") + TABPFN_MSG)

### 👀 Wow moment: how do they *see* the data?
Before the numbers, a picture. Below, each row is a model and each column a training-set size on the two-moons toy problem. Trees carve the plane into axis-aligned boxes. TabPFN's prior, learned from millions of synthetic datasets, yields smooth, sensible boundaries even from **16 points**. (If TabPFN is unavailable you'll see a logistic-regression row instead.)

> 🔮 **Predict-then-run:** at $n = 16$, which model will have the higher test accuracy?

In [ ]:
from sklearn.datasets import make_moons
Xmo, ymo = make_moons(1600, noise=0.25, random_state=SEED); Xmo = Xmo.astype(np.float32)
Xmo_te, ymo_te = Xmo[1000:], ymo[1000:]
SIZES = [16, 64, 256] if FAST else [16, 32, 64, 128, 256]
ROWS = {"HistGB (min_samples_leaf=3)": lambda: HistGradientBoostingClassifier(min_samples_leaf=3, random_state=SEED)}
if TABPFN_OK:
    ROWS["TabPFN v2 (no tuning)"] = lambda: make_tabpfn("clf")
else:
    ROWS["LogReg"] = lambda: make_pipeline(StandardScaler(), LogisticRegression())
GN = 50 if FAST else 80
gx, gy = np.meshgrid(np.linspace(-1.6, 2.6, GN), np.linspace(-1.1, 1.6, GN))
G = np.c_[gx.ravel(), gy.ravel()].astype(np.float32)
fig, axes = plt.subplots(len(ROWS), len(SIZES), figsize=(3.3 * len(SIZES), 3.0 * len(ROWS)), squeeze=False)
for r_i, (name, make) in enumerate(ROWS.items()):
    for c_i, n in enumerate(SIZES):
        idx = np.random.default_rng(n).choice(1000, n, replace=False)
        mdl = make().fit(Xmo[idx], ymo[idx])
        pz = mdl.predict_proba(G)[:, 1].reshape(gx.shape)
        acc = (mdl.predict_proba(Xmo_te).argmax(1) == ymo_te).mean()
        ax = axes[r_i, c_i]
        ax.contourf(gx, gy, pz, levels=np.linspace(0, 1, 21), cmap="RdYlBu_r", alpha=0.8)
        ax.contour(gx, gy, pz, levels=[0.5], colors=HIVE[1], linewidths=1.5)
        ax.scatter(*Xmo[idx].T, c=ymo[idx], cmap="RdYlBu_r", s=14, edgecolor="k", linewidth=0.4)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"n={n} · test acc {acc:.2f}", fontsize=10)
        if c_i == 0: ax.set_ylabel(name, fontsize=10)
plt.suptitle("Boxes vs. a learned prior: decision surfaces as the training set grows", y=1.01); plt.tight_layout(); plt.show()

### The contenders
* **Logistic regression** (standardised): the honest linear baseline.
* **HistGB default**: scikit-learn's LightGBM-style booster, out of the box.
* **HistGB tuned**: a random search over learning rate, leaves, min leaf size, L2 and iterations, with inner 3-fold CV on log loss. That's ~12–20 model fits per outer fold. This is what "tuned GBDT" means in papers (theirs use far bigger budgets).
* **TabPFN v2**: no tuning, one forward pass.

The datasets are small classics built into scikit-learn (no downloads), plus a noisy synthetic one. We use outer stratified K-fold and report ROC-AUC (one-vs-rest for multi-class), accuracy and log loss.

> 🔮 **Predict-then-run:** on datasets with a few hundred rows, which model wins on AUC? On log loss? Which is fastest *including tuning*?

In [ ]:
def tuned_hgb(n_iter=None):
    space = {"learning_rate": loguniform(0.01, 0.3), "max_leaf_nodes": randint(4, 48), "min_samples_leaf": randint(3, 40),
             "l2_regularization": loguniform(1e-4, 10.0), "max_iter": randint(50, 400)}
    return RandomizedSearchCV(HistGradientBoostingClassifier(random_state=SEED), space, n_iter=n_iter or (4 if FAST else 12),
                              cv=3, scoring="neg_log_loss", random_state=SEED, n_jobs=N_JOBS)

MODELS = {
    "LogReg": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "HistGB default": lambda: HistGradientBoostingClassifier(random_state=SEED),
    "HistGB tuned": tuned_hgb,
}
if TABPFN_OK:
    MODELS["TabPFN v2"] = lambda: make_tabpfn("clf")

def subsample(X, y, n, seed=SEED):
    if len(y) <= n: return X, y
    idx, _ = train_test_split(np.arange(len(y)), train_size=n, stratify=y, random_state=seed)
    return X[idx], y[idx]

DATASETS = {
    "breast_cancer (569×30, 2 cls)": load_breast_cancer(return_X_y=True),
    "wine (178×13, 3 cls)": load_wine(return_X_y=True),
}
if not FAST:
    DATASETS["digits (600×64, 10 cls)"] = subsample(*load_digits(return_X_y=True), 600)
    DATASETS["synthetic noisy (500×20, 2 cls)"] = make_classification(500, 20, n_informative=6, flip_y=0.1, class_sep=0.8, random_state=SEED)
N_SPLITS = 2 if FAST else 3

def run_benchmark(datasets, models, n_splits):
    rows = []
    for dname, (X, y) in datasets.items():
        X = X.astype(np.float32)
        for fold, (tr, te) in enumerate(StratifiedKFold(n_splits, shuffle=True, random_state=SEED).split(X, y)):
            for mname, make in models.items():
                try:
                    t0 = time.time(); mdl = make().fit(X[tr], y[tr]); P = mdl.predict_proba(X[te]); dt = time.time() - t0
                except Exception as e:
                    print(f"  ⚠️ {mname} failed on {dname}: {repr(e)[:120]}"); continue
                auc = roc_auc_score(y[te], P[:, 1]) if P.shape[1] == 2 else roc_auc_score(y[te], P, multi_class="ovr")
                rows.append({"dataset": dname, "model": mname, "fold": fold, "AUC": auc, "accuracy": accuracy_score(y[te], P.argmax(1)),
                             "log loss": log_loss(y[te], np.clip(P, 1e-7, 1), labels=np.unique(y)), "seconds": dt})
        print(f"done: {dname}")
    return pd.DataFrame(rows)

t0 = time.time()
bench = run_benchmark(DATASETS, MODELS, N_SPLITS)
print(f"benchmark wall time: {time.time() - t0:.0f}s")
summary = bench.groupby(["dataset", "model"])[["AUC", "accuracy", "log loss", "seconds"]].mean()
display(summary.style.format("{:.4f}").format("{:.2f}", subset=["seconds"]).highlight_max(subset=["AUC", "accuracy"], color="#FCE9A6")
        .highlight_min(subset=["log loss"], color="#FCE9A6"))

In [ ]:
piv_ll = summary["log loss"].unstack()
piv_t = summary["seconds"].unstack()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.3))
piv_ll.plot.bar(ax=axes[0], color=HIVE[:len(piv_ll.columns)], width=0.8, rot=0)
axes[0].set_title("log loss (lower is better)"); axes[0].set_xlabel("")
axes[0].set_xticklabels([t.get_text().split(" (")[0] for t in axes[0].get_xticklabels()])
piv_t.plot.bar(ax=axes[1], color=HIVE[:len(piv_t.columns)], width=0.8, rot=0, logy=True, legend=False)
axes[1].set_title("fit + predict time per fold, seconds (log)"); axes[1].set_xlabel("")
axes[1].set_xticklabels([t.get_text().split(" (")[0] for t in axes[1].get_xticklabels()])
plt.tight_layout(); plt.show()

### 🧪 Your Turn 5 — Average rank, the benchmark summary statistic
Papers summarise "who wins" across datasets with the **average rank** (1 = best on that dataset; ties get the average rank). Implement `average_ranks(bench, metric, higher_is_better)`.

*Hints:* `bench.groupby(["dataset", "model"])[metric].mean().unstack()` gives a dataset × model table; `.rank(axis=1, ascending=...)` ranks within each dataset.

In [ ]:
# 🧪 Your Turn 5
def average_ranks(bench, metric="AUC", higher_is_better=True):
    table = None   # TODO: dataset x model table of mean metric
    ranks = None   # TODO: rank models within each dataset (row)
    return ranks   # TODO: return the average rank per model, sorted ascending

In [ ]:
# ✅ Solution (peek only after trying!)
def average_ranks(bench, metric="AUC", higher_is_better=True):
    table = bench.groupby(["dataset", "model"])[metric].mean().unstack()
    ranks = table.rank(axis=1, ascending=not higher_is_better)
    return ranks.mean().sort_values()

In [ ]:
# ✔️ Checkpoint + summary table
ar = pd.DataFrame({"avg rank (AUC)": average_ranks(bench, "AUC", True),
                   "avg rank (log loss)": average_ranks(bench, "log loss", False)})
# ✔️ Checkpoint
n_models = bench["model"].nunique()
assert ar.values.min() >= 1 and ar.values.max() <= n_models
assert abs(ar["avg rank (AUC)"].sum() - n_models * (n_models + 1) / 2) < 1e-9
print("✔️ Average ranks computed (lower is better)")
ar.sort_values("avg rank (log loss)").round(2)

### Bonus: regression with `TabPFNRegressor`
TabPFN also does regression, and it outputs a full predictive *distribution*, not just a mean. Quick comparison on the diabetes data (442 rows):

In [ ]:
Xr, yr = load_diabetes(return_X_y=True); Xr = Xr.astype(np.float32)
reg_models = {
    "HistGB default": lambda: HistGradientBoostingRegressor(random_state=SEED),
    "HistGB tuned": lambda: RandomizedSearchCV(HistGradientBoostingRegressor(random_state=SEED),
        {"learning_rate": loguniform(0.01, 0.3), "max_leaf_nodes": randint(4, 32), "min_samples_leaf": randint(5, 50),
         "l2_regularization": loguniform(1e-4, 10.0), "max_iter": randint(50, 300)},
        n_iter=4 if FAST else 12, cv=3, random_state=SEED, n_jobs=N_JOBS),
}
if TABPFN_OK:
    reg_models["TabPFN v2"] = lambda: make_tabpfn("reg")
rows = []
for fold, (tr, te) in enumerate(KFold(N_SPLITS, shuffle=True, random_state=SEED).split(Xr)):
    for name, make in reg_models.items():
        try:
            t0 = time.time(); pred = make().fit(Xr[tr], yr[tr]).predict(Xr[te])
            rows.append({"model": name, "R²": r2_score(yr[te], pred), "seconds": time.time() - t0})
        except Exception as e:
            print(f"⚠️ {name} failed: {repr(e)[:150]}")
pd.DataFrame(rows).groupby("model").mean().round(3)

> **🍯 Key takeaway:** TabPFN "trains" with a single forward pass, using a prior learned from synthetic datasets. On small tables it is often as good as or better than a tuned GBDT, and it usually has the best log loss.

---
## Part 6 — 🍯 The Honey Yield Leaderboard (fun challenge)

Buzz runs 600 beehives and wants to predict which colonies will **thrive** this season. The hidden ground truth has nonlinear effects, an interaction or two, and a couple of pure-noise sensors. You get the 600 labelled hives. The leaderboard scores **ROC-AUC on 4,000 hidden hives**.

Rules: any model, any features, but you may only train on `X_hive`, `y_hive`. Baselines are already on the board. **Beat them all!** 🏆

In [ ]:
def _hive_frame(n, r):
    return pd.DataFrame({
        "temp_c": r.normal(22, 6, n), "humidity_pct": r.uniform(20, 95, n), "flower_density": r.gamma(2.0, 2.0, n),
        "hive_age_yrs": r.integers(0, 12, n).astype(float), "varroa_mites": r.poisson(3, n).astype(float),
        "queen_age_yrs": r.uniform(0, 5, n), "dist_to_water_km": r.exponential(1.5, n),
        "beekeeper_visits": r.poisson(4, n).astype(float), "sensor_a": r.normal(size=n), "sensor_b": r.normal(size=n)})

def _hive_logit(df):
    return (-0.02 * (df.temp_c - 24) ** 2 + 0.9 * np.log1p(df.flower_density) - 0.35 * df.varroa_mites
            + 0.8 * np.sin(df.humidity_pct / 12) - 0.25 * np.maximum(df.queen_age_yrs - 2.5, 0) * df.varroa_mites
            - 0.45 * df.dist_to_water_km + 0.25 * df.beekeeper_visits * (df.hive_age_yrs > 6))

_OFFSET = float(np.median(_hive_logit(_hive_frame(100_000, np.random.default_rng(999)))))
def make_hive(n, seed):
    r = np.random.default_rng(seed)
    df = _hive_frame(n, r)
    p = 1 / (1 + np.exp(-(_hive_logit(df) - _OFFSET)))
    return df.astype(np.float32), (r.uniform(size=n) < p).astype(int)

X_hive, y_hive = make_hive(600, seed=2026)
_X_hidden, _y_hidden = make_hive(4000, seed=7)        # 🙈 no peeking!
LEADERBOARD = []

def submit(name, model):
    t0 = time.time()
    model.fit(X_hive.values, y_hive)
    auc = roc_auc_score(_y_hidden, model.predict_proba(_X_hidden.values)[:, 1])
    LEADERBOARD.append({"team": name, "hidden AUC": auc, "seconds": time.time() - t0})
    board = pd.DataFrame(LEADERBOARD).sort_values("hidden AUC", ascending=False).reset_index(drop=True)
    board.index = [["🥇", "🥈", "🥉"][k] if k < 3 else str(k + 1) for k in range(len(board))]
    return board.round(4)

print(f"positives: {y_hive.mean():.2f} | features: {list(X_hive.columns)}")
submit("baseline: LogReg", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)))
submit("baseline: HistGB default", HistGradientBoostingClassifier(random_state=SEED))
if TABPFN_OK:
    submit("baseline: TabPFN v2", make_tabpfn("clf"))
display(submit("baseline: HistGB tuned", tuned_hgb()))

### 🧪 Your Turn 6 — Beat the board
Ideas, from easy to spicy:
* **Ensemble** TabPFN with a tuned GBDT by averaging probabilities. Different inductive biases make different errors.
* **Feature engineering for the linear model:** `SplineTransformer` before `LogisticRegression` turns it into a GAM that can bend.
* **Domain knowledge:** an interaction feature like `varroa_mites * queen_age_yrs`, or temperature distance from an ideal ~24 °C.
* Look at a SHAP beeswarm of the GBDT to *discover* which shapes and interactions matter. That's explanation used for model development!

In [ ]:
# 🧪 Your Turn 6
class AvgProba(BaseEstimator, ClassifierMixin):
    """Average predict_proba of several models (soft voting)."""
    def __init__(self, models, weights=None):
        self.models, self.weights = models, weights
    def fit(self, X, y):
        self.fitted_ = [clone(m).fit(X, y) for m in self.models]
        self.classes_ = np.unique(y)
        return self
    def predict_proba(self, X):
        return np.average([m.predict_proba(X) for m in self.fitted_], axis=0, weights=self.weights)

my_model = make_pipeline(StandardScaler(), LogisticRegression())   # TODO: build something better!
display(submit("my team (edit me)", my_model))

In [ ]:
# ✅ Solution (peek only after trying!): a GAM-ish spline model blended with TabPFN (or with a tuned GBDT if no TabPFN)
spline_lr = make_pipeline(SplineTransformer(n_knots=6, degree=3), LogisticRegression(C=0.5, max_iter=5000))
submit("spline GAM alone", spline_lr)   # smooth additive shapes alone already go a long way
members = [spline_lr, make_tabpfn("clf") if TABPFN_OK else tuned_hgb()]
board = submit("Buzz's soft-voting hive mind 🐝", AvgProba(members))
display(board)
best_baseline = max(row["hidden AUC"] for row in LEADERBOARD if row["team"].startswith("baseline"))
print(f"best baseline {best_baseline:.4f} vs ensemble {LEADERBOARD[-1]['hidden AUC']:.4f}")
print("Lesson: models with *different* inductive biases (smooth additive splines vs. learned-prior transformer / trees) blend best.")

> **🍯 Key takeaway:** Blending models with *different* inductive biases (splines, trees, transformers) is a reliable way to climb a leaderboard.

---
## Part 7 — When to use which? A field guide for 2026

| Situation | Reach for | Why |
|---|---|---|
| ≤ ~10k rows, need a strong model *fast*, GPU available | **TabPFN** (or another tabular FM) | no tuning, strong small-data accuracy, well-calibrated probabilities |
| 10k–10M+ rows, latency-critical serving | **GBDT** (LightGBM / XGBoost / CatBoost / HistGB) | cheap inference (microseconds per row), scales, mature tooling |
| Heavy high-cardinality categoricals | **CatBoost** / GBDT with target encoding | ordered target statistics, native handling |
| Regulated setting, need exact explanations | **GBDT + TreeSHAP**, or a GAM / EBM | fast exact SHAP; glass-box alternatives |
| Distribution shift, extrapolation needed | linear/GAM components + monitoring | trees and in-context learners don't extrapolate beyond the data |
| Very wide data (thousands of features) or huge context | GBDT, or FM variants built for scale | context length / memory limits of transformers |

**Trade-offs to remember**
* TabPFN moves the cost from **training** (≈ zero) to **inference**. Every prediction re-reads the training set as context, so high-volume, low-latency serving is costly compared with a GBDT.
* GBDTs need **tuning** to shine, but they're cheap at inference and easy to explain exactly with TreeSHAP.
* Any model can be explained with model-agnostic SHAP (`shap.PermutationExplainer` / `KernelExplainer`). It's just slower: exactly Part 1's sampling estimator.
* Benchmarks move fast. Living benchmarks (e.g., **TabArena**) track the current state of the art, so re-check before you bet a product on a leaderboard snapshot.

> **🍯 Key takeaway:** Default to tuned GBDTs at scale. Reach for a tabular foundation model for small data and fast iteration, and benchmark both on *your* data.

---
## 🏆 Boss Level
1. **KernelSHAP from scratch.** Implement the weighted-linear-regression formulation of Lundberg & Lee (2017): sample coalitions $z$, weight them by the Shapley kernel $\pi(z) = \frac{d-1}{\binom{d}{|z|}|z|(d-|z|)}$, and solve the weighted least squares problem with the efficiency constraint. Compare with your exact values from Part 1.
2. **Explain TabPFN.** Use `shap.PermutationExplainer` on TabPFN's `predict_proba` for the hive data (keep it to ~50 rows and a 50-row background). Do TabPFN and the tuned GBDT agree on the top-3 features? Where do their dependence plots differ?
3. **Scaling study.** For `n ∈ {100, 300, 1000, 3000}` training rows of the hive generator, plot hidden AUC and wall time for TabPFN and tuned HistGB. Where's the crossover on your hardware?

## 🍯 Key Takeaways
* **Shapley = the unique fair attribution** (efficiency, symmetry, dummy, additivity); exact cost is $2^d$.
* **TreeSHAP** is exact and polynomial for trees; **permutation sampling** works for any black box ($\propto 1/\sqrt{m}$ error).
* **The value function is a choice:** interventional ("true to the model") vs conditional ("true to the data").
* **Correlated features share credit.** Group them before you conclude a feature "doesn't matter".
* **SHAP ≠ causality.** It describes how the model uses inputs.
* **TabPFN** = transformer pre-trained on synthetic datasets that "fits" by in-context learning. It's strong and well calibrated on small data, and costly at inference.
* **Benchmark, then blend.** Tuned GBDTs remain the scalable default; diverse ensembles win leaderboards.

## 🎤 Explain it to a friend
Three friends share a taxi: Ana rides 2 km, Ben 6 km, Cy 10 km, and the route is shared as far as each one goes. Explain to a friend how to split the fare *fairly* by imagining every possible order in which the friends could have joined. Then connect it: SHAP does the same thing, with features as the friends and the prediction as the fare. Finally, explain how a model that has "practised" on millions of made-up spreadsheets could make good predictions on a brand-new one without any training.

## 💼 Interview corner
**Q1. Two highly correlated features both have small mean |SHAP|. Can you drop them?**
<details><summary>Answer</summary>

Not without checking. The model may be splitting the credit for one underlying signal between them. Sum their SHAP values into a group attribution, or drop one, retrain and compare validation performance. Dropping *both* could remove the signal entirely.
</details>

**Q2. What's the difference between interventional and observational (conditional) SHAP?**
<details><summary>Answer</summary>

Interventional SHAP fills the "missing" features from the background *marginal* distribution. That breaks feature dependence, so it is "true to the model" but can evaluate unrealistic, off-manifold inputs. Conditional/observational SHAP fills them in from $p(x_{\bar S} \mid x_S)$, which is "true to the data". The catch: it can give credit to features the model never uses, simply because they are correlated with used ones. Which one is right depends on the question you're asking.
</details>

**Q3. When would you pick TabPFN over XGBoost for a production model?**
<details><summary>Answer</summary>

Small datasets (roughly ≤ 10k rows and a few hundred features), when you need a strong, well-calibrated model quickly without a tuning budget, and when inference volume is modest or a GPU is available. Pick XGBoost/LightGBM for large data, strict latency or cost constraints on CPU, very wide data, and when you want cheap exact TreeSHAP explanations. Either way, benchmark on your data.
</details>

## 🐝 Fun fact
Lloyd Shapley introduced the Shapley value in 1953, and he did win a Nobel: the 2012 Nobel Memorial Prize in Economic Sciences, shared with Alvin Roth. The prize, though, was for the theory of stable allocations and market design (think the Gale–Shapley matching algorithm used for medical residencies), not for the value that now explains millions of ML predictions.

## 📚 Further reading
* L. S. Shapley (1953). *A Value for n-Person Games.* Contributions to the Theory of Games II.
* S. M. Lundberg & S.-I. Lee (2017). *A Unified Approach to Interpreting Model Predictions.* NeurIPS.
* S. M. Lundberg et al. (2020). *From local explanations to global understanding with explainable AI for trees.* Nature Machine Intelligence 2.
* E. Štrumbelj & I. Kononenko (2014). *Explaining prediction models and individual predictions with feature contributions.* Knowledge and Information Systems 41.
* D. Janzing, L. Minorics, P. Blöbaum (2020). *Feature relevance quantification in explainable AI: A causal problem.* AISTATS.
* I. E. Kumar, S. Venkatasubramanian, C. Scheidegger, S. Friedler (2020). *Problems with Shapley-value-based explanations as feature importance measures.* ICML.
* H. Chen, J. D. Janizek, S. Lundberg, S.-I. Lee (2020). *True to the Model or True to the Data?* arXiv preprint.
* S. Müller, N. Hollmann, S. Pineda Arango, J. Grabocka, F. Hutter (2022). *Transformers Can Do Bayesian Inference.* ICLR.
* N. Hollmann, S. Müller, K. Eggensperger, F. Hutter (2023). *TabPFN: A Transformer That Solves Small Tabular Classification Problems in a Second.* ICLR.
* N. Hollmann et al. (2025). *Accurate predictions on small data with a tabular foundation model.* Nature 637.
* L. Grinsztajn, E. Oyallon, G. Varoquaux (2022). *Why do tree-based models still outperform deep learning on typical tabular data?* NeurIPS Datasets & Benchmarks.
* C. Molnar. *Interpretable Machine Learning* (online book), chapters on Shapley values and SHAP.

🐝 *Buzz says: that's Day 1 in the bag. You can build boosters, make their probabilities honest, explain them, and know when a foundation model should replace them. See you on Day 2!*